In [ ]:
from __future__ import annotations

import math
from dataclasses import asdict, dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def veilige_del(a: float, b: float, fallback: float = float("nan")) -> float:
    return fallback if abs(b) < 1e-16 else a / b


def mm(waarde_m: float) -> float:
    return 1000.0 * waarde_m


def m_uit_mm(waarde_mm: float) -> float:
    return waarde_mm / 1000.0


def rond_naar_10_mm_boven(waarde_mm: float) -> int:
    return int(math.ceil(waarde_mm / 10.0) * 10.0)


def unieke_reeks_mm(*waarden: int, minimum: int = 0) -> Tuple[int, ...]:
    out = sorted({int(v) for v in waarden if int(v) >= minimum})
    return tuple(out)


def toon_dataframe(naam: str, df: pd.DataFrame, max_rijen: int = 20) -> None:
    print(f"\n=== {naam.upper()} ===")
    if len(df) > max_rijen:
        print(df.head(max_rijen).to_string(index=False))
        print(f"... ({len(df)} rijen in totaal)")
    else:
        print(df.to_string(index=False))


try:
    resultaten_interface
except NameError:
    resultaten_interface = {
        "export_voor_volgende_modules": {
            "Fz_max_N": 1179.3,               # Tom v19
            "Mx_max_Nm": 40.95,              # Tom v19.0443845
            "My_max_Nm": 493.68779791405746,
            "Mz_max_Nm": 1.38996,
        }
    }
try:
    resultaten_spindel
except NameError:
    resultaten_spindel = {"export_voor_volgende_modules": {"nominale_diameter_spindel_m": 0.040, "m1_schacht_totaal_m": 2.478, "m1_slag_m": 2.200}}

try:
    resultaten_motor
except NameError:
    resultaten_motor = {"export_voor_volgende_modules": {"motor_flange_mm": 130.0, "motor_body_diameter_mm": 130.0, "motor_body_length_mm": 180.0, "motor_shaft_diameter_mm": 19.0}}

try:
    resultaten_koppeling_notebook
except NameError:
    resultaten_koppeling_notebook = {"configuratie_koppeling": {"diameter_motorkant_m": 0.019, "diameter_lastkant_m": 0.024}, "export_voor_volgende_modules": {"d_min_torsie_mm": 8.0}}


@dataclass(frozen=True)
class Materiaal:
    naam: str
    E_Pa: float
    rho_kg_m3: float
    sigma_vloei_Pa: float
    nu: float = 0.30

    @property
    def G_Pa(self) -> float:
        return self.E_Pa / (2.0 * (1.0 + self.nu))


MATERIALEN: Dict[str, Materiaal] = {
    "staal_s355": Materiaal("Staal S355", 210e9, 7850.0, 355e6, 0.30),
    "aluminium_6082_t6": Materiaal("Aluminium 6082-T6", 70e9, 2700.0, 260e6, 0.33),
}


@dataclass(frozen=True)
class Boutpatroon:
    dx_m: float
    dy_m: float

    def punten(self) -> np.ndarray:
        return np.array([
            [-self.dx_m / 2.0, -self.dy_m / 2.0],
            [ self.dx_m / 2.0, -self.dy_m / 2.0],
            [-self.dx_m / 2.0,  self.dy_m / 2.0],
            [ self.dx_m / 2.0,  self.dy_m / 2.0],
        ], dtype=float)


@dataclass(frozen=True)
class Plaatconcept:
    materiaal: str
    breedte_m: float
    hoogte_m: float
    dikte_m: float
    aantal_ribben: int
    hoogte_rib_m: float
    dikte_rib_m: float
    patroon_m2: Boutpatroon
    patroon_slede: Boutpatroon


@dataclass(frozen=True)
class ConfiguratieMontageplaat:
    breedte_max_plaat_m: float
    hoogte_max_plaat_m: float
    breedte_sledezone_m: float
    hoogte_sledezone_m: float
    breedte_keepout_spindelmoer_m: float
    hoogte_keepout_spindelmoer_m: float
    gatdiameter_mm: float = 11.0
    factor_randafstand: float = 1.8
    factor_min_steek: float = 3.0
    veiligheidsfactor_plaat: float = 2.0
    toegelaten_doorbuiging_plaat_centrum_mm: float = 0.20
    toegelaten_rotatie_plaat_graden: float = 0.10
    toegelaten_massa_plaat_kg: float = 35.0
    belastingsfactor: float = 1.25
    breedtes_plaat_mm: Tuple[int, ...] = (400, 420, 440)
    hoogtes_plaat_mm: Tuple[int, ...] = (320, 340, 360)
    diktes_plaat_mm: Tuple[int, ...] = (12, 15, 18, 20, 25)
    aantallen_ribben_plaat: Tuple[int, ...] = (0, 2, 3, 4)
    hoogtes_rib_plaat_mm: Tuple[int, ...] = (0, 20, 30, 40, 50)
    diktes_rib_plaat_mm: Tuple[int, ...] = (0, 5, 6, 8)
    dx_m2_bouten_mm: Tuple[int, ...] = (140, 160, 180, 200)
    dy_m2_bouten_mm: Tuple[int, ...] = (100, 120, 140, 160)
    dx_slede_bouten_mm: Tuple[int, ...] = (120, 140, 160, 180)
    dy_slede_bouten_mm: Tuple[int, ...] = (80, 100, 120, 140)
    aantal_topkandidaten_plaat: int = 15
    voorkeursmateriaal_plaat: str = "staal_s355"
    voorkeurs_breedte_plaat_mm: float = 440.0
    voorkeurs_hoogte_plaat_mm: float = 360.0
    voorkeurs_dikte_plaat_mm: float = 20.0
    voorkeurs_aantal_ribben: int = 2
    voorkeurs_hoogte_rib_mm: float = 40.0
    voorkeurs_dikte_rib_mm: float = 6.0
    voorkeurs_dx_m2_bouten_mm: float = 180.0
    voorkeurs_dy_m2_bouten_mm: float = 120.0
    voorkeurs_dx_slede_bouten_mm: float = 160.0
    voorkeurs_dy_slede_bouten_mm: float = 100.0


def maak_interface_last_plaat() -> Dict[str, float]:
    exp = resultaten_interface["export_voor_volgende_modules"]
    return {"Fz_N": float(exp["Fz_max_N"]), "Mx_Nm": float(exp["Mx_max_Nm"]), "My_Nm": float(exp["My_max_Nm"]), "Mz_Nm": float(exp["Mz_max_Nm"])}


def maak_afgeleide_plaatconfiguratie() -> ConfiguratieMontageplaat:
    sp = resultaten_spindel["export_voor_volgende_modules"]
    mot = resultaten_motor["export_voor_volgende_modules"]
    kop_cfg = resultaten_koppeling_notebook.get("configuratie_koppeling", {})
    d_spindel_mm = 1000.0 * float(sp.get("nominale_diameter_spindel_m", 0.040))
    motor_flange_mm = float(mot.get("motor_flange_mm", 130.0))
    motor_diameter_mm = float(mot.get("motor_body_diameter_mm", motor_flange_mm))
    koppeling_d_mm = 1000.0 * max(float(kop_cfg.get("diameter_motorkant_m", 0.019)), float(kop_cfg.get("diameter_lastkant_m", 0.024))) * 2.5

    breedte_sledezone_mm = 320.0
    hoogte_sledezone_mm = 240.0
    keepout_breedte_mm = max(160.0, 3.8 * d_spindel_mm + 20.0, motor_diameter_mm)
    keepout_hoogte_mm = max(140.0, 3.2 * d_spindel_mm + 20.0, koppeling_d_mm)

    req_breedte_mm = max(breedte_sledezone_mm + 100.0, keepout_breedte_mm + 180.0, motor_flange_mm + 220.0)
    req_hoogte_mm = max(hoogte_sledezone_mm + 100.0, keepout_hoogte_mm + 180.0, motor_flange_mm + 180.0)
    voorkeurs_breedte_mm = rond_naar_10_mm_boven(req_breedte_mm)
    voorkeurs_hoogte_mm = rond_naar_10_mm_boven(req_hoogte_mm)

    breedtes = unieke_reeks_mm(voorkeurs_breedte_mm - 40, voorkeurs_breedte_mm - 20, voorkeurs_breedte_mm, minimum=300)
    hoogtes = unieke_reeks_mm(voorkeurs_hoogte_mm - 40, voorkeurs_hoogte_mm - 20, voorkeurs_hoogte_mm, minimum=260)

    return ConfiguratieMontageplaat(
        breedte_max_plaat_m=m_uit_mm(voorkeurs_breedte_mm),
        hoogte_max_plaat_m=m_uit_mm(voorkeurs_hoogte_mm),
        breedte_sledezone_m=m_uit_mm(breedte_sledezone_mm),
        hoogte_sledezone_m=m_uit_mm(hoogte_sledezone_mm),
        breedte_keepout_spindelmoer_m=m_uit_mm(keepout_breedte_mm),
        hoogte_keepout_spindelmoer_m=m_uit_mm(keepout_hoogte_mm),
        breedtes_plaat_mm=breedtes,
        hoogtes_plaat_mm=hoogtes,
        voorkeurs_breedte_plaat_mm=voorkeurs_breedte_mm,
        voorkeurs_hoogte_plaat_mm=voorkeurs_hoogte_mm,
    )


def sectie_eigenschappen_plaat(concept: Plaatconcept) -> Dict[str, float]:
    materiaal = MATERIALEN[concept.materiaal]
    b = concept.breedte_m
    h = concept.hoogte_m
    t = concept.dikte_m
    I_plaat_x = t * h ** 3 / 12.0
    I_plaat_y = h * b ** 3 / 12.0
    I_ribben_x = 0.0
    I_ribben_y = 0.0
    A_ribben = 0.0
    if concept.aantal_ribben > 0 and concept.hoogte_rib_m > 0 and concept.dikte_rib_m > 0:
        A_rib = concept.dikte_rib_m * concept.hoogte_rib_m
        I_rib_lokaal_x = concept.dikte_rib_m * concept.hoogte_rib_m ** 3 / 12.0
        I_rib_lokaal_y = concept.hoogte_rib_m * concept.dikte_rib_m ** 3 / 12.0
        afstand_neutraal = t / 2.0 + concept.hoogte_rib_m / 2.0
        x_posities = np.linspace(-0.35 * b, 0.35 * b, concept.aantal_ribben)
        for x_i in x_posities:
            I_ribben_x += I_rib_lokaal_x + A_rib * afstand_neutraal ** 2
            I_ribben_y += I_rib_lokaal_y + A_rib * (x_i ** 2 + afstand_neutraal ** 2)
        A_ribben = concept.aantal_ribben * A_rib
    I_eq_x = I_plaat_x + I_ribben_x
    I_eq_y = I_plaat_y + I_ribben_y
    W_eq_x = veilige_del(I_eq_x, h / 2.0, float("nan"))
    W_eq_y = veilige_del(I_eq_y, b / 2.0, float("nan"))
    massa_plaat = b * h * t * materiaal.rho_kg_m3
    massa_ribben = A_ribben * h * materiaal.rho_kg_m3 if concept.aantal_ribben > 0 else 0.0
    return {"I_eq_x_m4": I_eq_x, "I_eq_y_m4": I_eq_y, "W_eq_x_m3": W_eq_x, "W_eq_y_m3": W_eq_y, "massa_totaal_kg": massa_plaat + massa_ribben}


def patroon_binnen_zone(zone_breedte_m: float, zone_hoogte_m: float, patroon: Boutpatroon, rand_m: float) -> bool:
    punten = patroon.punten()
    if np.any(np.abs(punten[:, 0]) > (zone_breedte_m / 2.0 - rand_m)):
        return False
    if np.any(np.abs(punten[:, 1]) > (zone_hoogte_m / 2.0 - rand_m)):
        return False
    return True


def randafstand_ok(zone_breedte_m: float, zone_hoogte_m: float, patroon: Boutpatroon, gat_diameter_m: float, factor: float) -> bool:
    return patroon_binnen_zone(zone_breedte_m, zone_hoogte_m, patroon, factor * gat_diameter_m)


def steek_ok(patroon: Boutpatroon, gat_diameter_m: float, factor: float) -> bool:
    minimum = factor * gat_diameter_m
    return patroon.dx_m >= minimum and patroon.dy_m >= minimum


def buiten_keepout(patroon: Boutpatroon, breedte_keepout_m: float, hoogte_keepout_m: float, gat_diameter_m: float) -> bool:
    straal = gat_diameter_m / 2.0
    for x, y in patroon.punten():
        if abs(x) < (breedte_keepout_m / 2.0 + straal) and abs(y) < (hoogte_keepout_m / 2.0 + straal):
            return False
    return True


def patronen_voldoende_uit_elkaar(p1: Boutpatroon, p2: Boutpatroon, minimum_hartafstand_m: float) -> bool:
    for a in p1.punten():
        for b in p2.punten():
            if np.linalg.norm(a - b) < minimum_hartafstand_m:
                return False
    return True


def boutgroep_krachten(patroon: Boutpatroon, Fx_N: float, Fy_N: float, Mz_Nm: float) -> np.ndarray:
    punten = patroon.punten()
    n = len(punten)
    F_recht = np.array([Fx_N / n, Fy_N / n], dtype=float)
    som_r2 = np.sum(np.sum(punten ** 2, axis=1))
    if som_r2 <= 0:
        return np.full(n, np.nan)
    uit = []
    for x, y in punten:
        F_moment = np.array([-Mz_Nm * y / som_r2, Mz_Nm * x / som_r2], dtype=float)
        F_totaal = F_recht + F_moment
        uit.append(np.linalg.norm(F_totaal))
    return np.array(uit, dtype=float)


def doorbuiging_eenvoudig_ondersteund_puntlast(P_N: float, L_m: float, x_m: float, E_Pa: float, I_m4: float) -> float:
    if min(L_m, E_Pa, I_m4) <= 0.0:
        return float("nan")
    a = x_m
    b = L_m - x_m
    return P_N * a * b * (L_m ** 2 - a ** 2 - b ** 2) / (6.0 * E_Pa * I_m4 * L_m)


def evalueer_plaat(concept: Plaatconcept, cfg: ConfiguratieMontageplaat, interface_lasten: Dict[str, float], belastingsfactor: float = 1.25) -> Dict[str, float]:
    materiaal = MATERIALEN[concept.materiaal]
    sectie = sectie_eigenschappen_plaat(concept)
    Fz = belastingsfactor * interface_lasten["Fz_N"]
    Mx = belastingsfactor * interface_lasten["Mx_Nm"]
    My = belastingsfactor * interface_lasten["My_Nm"]
    Mz = belastingsfactor * interface_lasten["Mz_Nm"]
    sigma_x = veilige_del(abs(Mx), sectie["W_eq_x_m3"], 0.0)
    sigma_y = veilige_del(abs(My), sectie["W_eq_y_m3"], 0.0)
    sigma_eq = sigma_x + sigma_y
    sigma_toeg = materiaal.sigma_vloei_Pa / cfg.veiligheidsfactor_plaat
    delta_x = doorbuiging_eenvoudig_ondersteund_puntlast(Fz, concept.hoogte_m, concept.hoogte_m / 2.0, materiaal.E_Pa, sectie["I_eq_x_m4"])
    delta_y = doorbuiging_eenvoudig_ondersteund_puntlast(Fz, concept.breedte_m, concept.breedte_m / 2.0, materiaal.E_Pa, sectie["I_eq_y_m4"])
    delta_centrum = max(abs(delta_x), abs(delta_y))
    theta_x = veilige_del(abs(Mx) * concept.hoogte_m, 2.0 * materiaal.E_Pa * sectie["I_eq_x_m4"], 0.0)
    theta_y = veilige_del(abs(My) * concept.breedte_m, 2.0 * materiaal.E_Pa * sectie["I_eq_y_m4"], 0.0)
    theta_totaal = theta_x + theta_y
    kracht_m2 = boutgroep_krachten(concept.patroon_m2, 0.0, 0.0, Mz)
    kracht_slede = boutgroep_krachten(concept.patroon_slede, 0.0, 0.0, Mz)
    return {
        "sigma_eq_Pa": sigma_eq,
        "sigma_toeg_Pa": sigma_toeg,
        "benutting_spanning": veilige_del(sigma_eq, sigma_toeg, float("nan")),
        "doorbuiging_centrum_mm": 1000.0 * delta_centrum,
        "benutting_doorbuiging": veilige_del(1000.0 * delta_centrum, cfg.toegelaten_doorbuiging_plaat_centrum_mm, float("nan")),
        "rotatie_graden": theta_totaal * 180.0 / math.pi,
        "benutting_rotatie": veilige_del(theta_totaal * 180.0 / math.pi, cfg.toegelaten_rotatie_plaat_graden, float("nan")),
        "massa_totaal_kg": sectie["massa_totaal_kg"],
        "benutting_massa": veilige_del(sectie["massa_totaal_kg"], cfg.toegelaten_massa_plaat_kg, float("nan")),
        "I_eq_x_m4": sectie["I_eq_x_m4"],
        "I_eq_y_m4": sectie["I_eq_y_m4"],
        "W_eq_x_m3": sectie["W_eq_x_m3"],
        "W_eq_y_m3": sectie["W_eq_y_m3"],
        "max_boutkracht_m2_N": float(np.nanmax(kracht_m2)),
        "max_boutkracht_slede_N": float(np.nanmax(kracht_slede)),
    }


def evalueer_volledig_plaatconcept(concept: Plaatconcept, cfg: ConfiguratieMontageplaat, interface_lasten: Dict[str, float]) -> Dict[str, object]:
    gat_diameter_m = m_uit_mm(cfg.gatdiameter_mm)
    minimum_tussenruimte = 2.5 * gat_diameter_m
    redenen: List[str] = []
    haalbaar = True
    if concept.breedte_m > cfg.breedte_max_plaat_m:
        haalbaar = False
        redenen.append("plaat te breed voor envelop")
    if concept.hoogte_m > cfg.hoogte_max_plaat_m:
        haalbaar = False
        redenen.append("plaat te hoog voor envelop")
    if not randafstand_ok(concept.breedte_m, concept.hoogte_m, concept.patroon_m2, gat_diameter_m, cfg.factor_randafstand):
        haalbaar = False
        redenen.append("M2-boutpatroon faalt randafstand")
    if not randafstand_ok(cfg.breedte_sledezone_m, cfg.hoogte_sledezone_m, concept.patroon_slede, gat_diameter_m, cfg.factor_randafstand):
        haalbaar = False
        redenen.append("sledeboutpatroon faalt randafstand")
    if not steek_ok(concept.patroon_m2, gat_diameter_m, cfg.factor_min_steek):
        haalbaar = False
        redenen.append("M2-boutpatroon faalt minimum steek")
    if not steek_ok(concept.patroon_slede, gat_diameter_m, cfg.factor_min_steek):
        haalbaar = False
        redenen.append("sledeboutpatroon faalt minimum steek")
    if not buiten_keepout(concept.patroon_m2, cfg.breedte_keepout_spindelmoer_m, cfg.hoogte_keepout_spindelmoer_m, gat_diameter_m):
        haalbaar = False
        redenen.append("M2-boutpatroon raakt keep-out")
    if not buiten_keepout(concept.patroon_slede, cfg.breedte_keepout_spindelmoer_m, cfg.hoogte_keepout_spindelmoer_m, gat_diameter_m):
        haalbaar = False
        redenen.append("sledeboutpatroon raakt keep-out")
    if not patronen_voldoende_uit_elkaar(concept.patroon_m2, concept.patroon_slede, minimum_tussenruimte):
        haalbaar = False
        redenen.append("boutpatronen liggen te dicht bij elkaar")
    respons = evalueer_plaat(concept, cfg, interface_lasten, cfg.belastingsfactor)
    if respons["sigma_eq_Pa"] > respons["sigma_toeg_Pa"]:
        haalbaar = False
        redenen.append("spanning te hoog")
    if respons["doorbuiging_centrum_mm"] > cfg.toegelaten_doorbuiging_plaat_centrum_mm:
        haalbaar = False
        redenen.append("doorbuiging te hoog")
    if respons["rotatie_graden"] > cfg.toegelaten_rotatie_plaat_graden:
        haalbaar = False
        redenen.append("rotatie te hoog")
    if respons["massa_totaal_kg"] > cfg.toegelaten_massa_plaat_kg:
        haalbaar = False
        redenen.append("plaatmassa te hoog")
    doel = 1.0 * respons["benutting_massa"] + 5000.0 * respons["benutting_spanning"] + 3000.0 * respons["benutting_doorbuiging"] + 3000.0 * respons["benutting_rotatie"] + (0.0 if haalbaar else 1e6)
    return {
        "haalbaar": haalbaar,
        "redenen": redenen,
        "doelfunctie": doel,
        "materiaal": concept.materiaal,
        "breedte_mm": mm(concept.breedte_m),
        "hoogte_mm": mm(concept.hoogte_m),
        "dikte_mm": mm(concept.dikte_m),
        "aantal_ribben": concept.aantal_ribben,
        "hoogte_rib_mm": mm(concept.hoogte_rib_m),
        "dikte_rib_mm": mm(concept.dikte_rib_m),
        "dx_m2_bouten_mm": mm(concept.patroon_m2.dx_m),
        "dy_m2_bouten_mm": mm(concept.patroon_m2.dy_m),
        "dx_slede_bouten_mm": mm(concept.patroon_slede.dx_m),
        "dy_slede_bouten_mm": mm(concept.patroon_slede.dy_m),
        **respons,
    }


def optimaliseer_montageplaat(cfg: ConfiguratieMontageplaat, interface_lasten: Dict[str, float]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rijen = []
    for materiaal in [cfg.voorkeursmateriaal_plaat]:
        for b_mm in cfg.breedtes_plaat_mm:
            for h_mm in cfg.hoogtes_plaat_mm:
                for t_mm in cfg.diktes_plaat_mm:
                    for n_rib in cfg.aantallen_ribben_plaat:
                        for hr_mm in cfg.hoogtes_rib_plaat_mm:
                            for tr_mm in cfg.diktes_rib_plaat_mm:
                                if n_rib == 0 and (hr_mm != 0 or tr_mm != 0):
                                    continue
                                if n_rib > 0 and (hr_mm == 0 or tr_mm == 0):
                                    continue
                                for dx_m2 in cfg.dx_m2_bouten_mm:
                                    for dy_m2 in cfg.dy_m2_bouten_mm:
                                        for dx_sl in cfg.dx_slede_bouten_mm:
                                            for dy_sl in cfg.dy_slede_bouten_mm:
                                                concept = Plaatconcept(materiaal, m_uit_mm(b_mm), m_uit_mm(h_mm), m_uit_mm(t_mm), n_rib, m_uit_mm(hr_mm), m_uit_mm(tr_mm), Boutpatroon(m_uit_mm(dx_m2), m_uit_mm(dy_m2)), Boutpatroon(m_uit_mm(dx_sl), m_uit_mm(dy_sl)))
                                                rijen.append(evalueer_volledig_plaatconcept(concept, cfg, interface_lasten))
    df = pd.DataFrame(rijen).sort_values(["haalbaar", "doelfunctie"], ascending=[False, True]).reset_index(drop=True)
    haalbaar = df[df["haalbaar"]].reset_index(drop=True)
    return df, haalbaar


def maak_voorkeursplaat(cfg: ConfiguratieMontageplaat) -> Plaatconcept:
    return Plaatconcept(cfg.voorkeursmateriaal_plaat, m_uit_mm(cfg.voorkeurs_breedte_plaat_mm), m_uit_mm(cfg.voorkeurs_hoogte_plaat_mm), m_uit_mm(cfg.voorkeurs_dikte_plaat_mm), cfg.voorkeurs_aantal_ribben, m_uit_mm(cfg.voorkeurs_hoogte_rib_mm), m_uit_mm(cfg.voorkeurs_dikte_rib_mm), Boutpatroon(m_uit_mm(cfg.voorkeurs_dx_m2_bouten_mm), m_uit_mm(cfg.voorkeurs_dy_m2_bouten_mm)), Boutpatroon(m_uit_mm(cfg.voorkeurs_dx_slede_bouten_mm), m_uit_mm(cfg.voorkeurs_dy_slede_bouten_mm)))


def maak_overzicht_voorkeursplaat(plaat_voorkeur: Dict[str, object]) -> pd.DataFrame:
    return pd.DataFrame([
        ["breedte plaat", plaat_voorkeur["breedte_mm"], "mm"],
        ["hoogte plaat", plaat_voorkeur["hoogte_mm"], "mm"],
        ["dikte plaat", plaat_voorkeur["dikte_mm"], "mm"],
        ["massa totaal", plaat_voorkeur["massa_totaal_kg"], "kg"],
        ["doorbuiging centrum", plaat_voorkeur["doorbuiging_centrum_mm"], "mm"],
        ["rotatie", plaat_voorkeur["rotatie_graden"], "deg"],
        ["spanning equivalent", plaat_voorkeur["sigma_eq_Pa"] / 1e6, "MPa"],
        ["max boutkracht M2", plaat_voorkeur["max_boutkracht_m2_N"], "N"],
        ["max boutkracht slede", plaat_voorkeur["max_boutkracht_slede_N"], "N"],
        ["haalbaar", plaat_voorkeur["haalbaar"], "-"],
    ], columns=["grootheid", "waarde", "eenheid"])


def lokale_studie_voorkeursplaat(cfg: ConfiguratieMontageplaat, interface_lasten: Dict[str, float], concept: Plaatconcept) -> pd.DataFrame:
    rijen = []
    for t_mm in (12, 15, 18, 20, 25):
        for hr_mm in (0, 20, 30, 40, 50):
            test = Plaatconcept(concept.materiaal, concept.breedte_m, concept.hoogte_m, m_uit_mm(t_mm), concept.aantal_ribben if hr_mm > 0 else 0, m_uit_mm(hr_mm), concept.dikte_rib_m if hr_mm > 0 else 0.0, concept.patroon_m2, concept.patroon_slede)
            rijen.append(evalueer_volledig_plaatconcept(test, cfg, interface_lasten))
    return pd.DataFrame(rijen).sort_values("doelfunctie").reset_index(drop=True)


def plot_massa_tegen_doorbuiging_plaat(df: pd.DataFrame) -> None:
    plt.figure(figsize=(7, 4))
    plt.scatter(df["massa_totaal_kg"], df["doorbuiging_centrum_mm"])
    plt.xlabel("massa [kg]")
    plt.ylabel("doorbuiging [mm]")
    plt.title("Trade-off montageplaat")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_layout_voorkeursplaat(concept: Plaatconcept, cfg: ConfiguratieMontageplaat) -> None:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.add_patch(plt.Rectangle((-concept.breedte_m / 2.0, -concept.hoogte_m / 2.0), concept.breedte_m, concept.hoogte_m, fill=False, linewidth=2.0))
    ax.add_patch(plt.Rectangle((-cfg.breedte_sledezone_m / 2.0, -cfg.hoogte_sledezone_m / 2.0), cfg.breedte_sledezone_m, cfg.hoogte_sledezone_m, fill=False, linestyle="--"))
    ax.add_patch(plt.Rectangle((-cfg.breedte_keepout_spindelmoer_m / 2.0, -cfg.hoogte_keepout_spindelmoer_m / 2.0), cfg.breedte_keepout_spindelmoer_m, cfg.hoogte_keepout_spindelmoer_m, alpha=0.25))
    p1 = concept.patroon_m2.punten()
    p2 = concept.patroon_slede.punten()
    ax.scatter(p1[:, 0], p1[:, 1], s=80, marker="o", label="M2-bouten")
    ax.scatter(p2[:, 0], p2[:, 1], s=80, marker="s", label="sledebouten")
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title("Voorkeursplaat - layout")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()

cfg_plaat = maak_afgeleide_plaatconfiguratie()
interface_lasten = maak_interface_last_plaat()
plaat_alle, plaat_haalbaar = optimaliseer_montageplaat(cfg_plaat, interface_lasten)
voorkeursplaat = maak_voorkeursplaat(cfg_plaat)
plaat_voorkeur = evalueer_volledig_plaatconcept(voorkeursplaat, cfg_plaat, interface_lasten)
plaat_lokale_studie = lokale_studie_voorkeursplaat(cfg_plaat, interface_lasten, voorkeursplaat)
overzicht_voorkeursplaat = maak_overzicht_voorkeursplaat(plaat_voorkeur)

print("\n=== INTERFACE LASTEN PLAAT ===")
print(pd.Series(interface_lasten).to_string())
if len(plaat_haalbaar) > 0:
    print("\n=== BESTE HAALBARE MONTAGEPLAAT ===")
    print(plaat_haalbaar.iloc[0].to_string())
    toon_dataframe("top montageplaatconcepten", plaat_haalbaar.head(15))
print("\n=== VALIDATIE VOORKEURSPLAAT ===")
print(pd.Series(plaat_voorkeur).to_string())
toon_dataframe("overzicht voorkeursplaat", overzicht_voorkeursplaat)
toon_dataframe("lokale studie voorkeursplaat", plaat_lokale_studie.head(20))

resultaten_montageplaat = {
    "configuratie_plaat": asdict(cfg_plaat),
    "interface_lasten": interface_lasten.copy(),
    "plaat_alle": plaat_alle.copy(),
    "plaat_haalbaar": plaat_haalbaar.copy(),
    "plaat_voorkeur": plaat_voorkeur.copy(),
    "plaat_lokale_studie": plaat_lokale_studie.copy(),
    "overzicht_voorkeursplaat": overzicht_voorkeursplaat.copy(),
    "export_voor_volgende_modules": {
        "voorkeursplaat_haalbaar": bool(plaat_voorkeur["haalbaar"]),
        "voorkeursplaat_breedte_mm": float(plaat_voorkeur["breedte_mm"]),
        "voorkeursplaat_hoogte_mm": float(plaat_voorkeur["hoogte_mm"]),
        "voorkeursplaat_dikte_mm": float(plaat_voorkeur["dikte_mm"]),
        "voorkeursplaat_massa_kg": float(plaat_voorkeur["massa_totaal_kg"]),
        "voorkeursplaat_doorbuiging_mm": float(plaat_voorkeur["doorbuiging_centrum_mm"]),
        "voorkeursplaat_rotatie_graden": float(plaat_voorkeur["rotatie_graden"]),
        "voorkeursplaat_spanning_MPa": float(plaat_voorkeur["sigma_eq_Pa"] / 1e6),
        "plaat_keepout_breedte_mm": float(mm(cfg_plaat.breedte_keepout_spindelmoer_m)),
        "plaat_keepout_hoogte_mm": float(mm(cfg_plaat.hoogte_keepout_spindelmoer_m)),
    },
}

print(resultaten_montageplaat["export_voor_volgende_modules"])
